In [1]:
import pandas as pd

df = pd.read_csv("./data/features.csv")

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 982 entries, 0 to 981
Data columns (total 27 columns):
 #   Column                                    Non-Null Count  Dtype  
---  ------                                    --------------  -----  
 0   Name                                      982 non-null    object 
 1   HRID                                      982 non-null    int64  
 2   Address                                   982 non-null    object 
 3   Community                                 982 non-null    object 
 4   Ward                                      982 non-null    int64  
 5   Resource Type                             969 non-null    object 
 6   Year of Construction                      978 non-null    object 
 7   Development Era                           982 non-null    object 
 8   Architectural Style                       806 non-null    object 
 9   Architect                                 759 non-null    object 
 10  Builder                               

In [2]:
col_to_drop = ["HRID", "Address", "Alternate Names", "Builder", "Architect"]

df.drop(columns=df.iloc[:,21:27], inplace=True)


df.drop(columns=col_to_drop, inplace=True)

df["Original Use"].fillna(df["Original Use"].mode()[0], inplace=True)





C:\Users\canmo\AppData\Local\Temp\ipykernel_12844\3546907731.py:8: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["Original Use"].fillna(df["Original Use"].mode()[0], inplace=True)


In [3]:
import re

# original (to compare and see)
df_before = df["Year of Construction"].copy()

# fillna 
def extract_median_year(era):
    years = re.findall(r"\d{4}", str(era))
    if len(years) == 2:
        return (int(years[0]) + int(years[1])) // 2
    elif len(years) == 1:
        return int(years[0])
    return None

df["Year of Construction"] = df["Year of Construction"].fillna(
    df["Development Era"].map(extract_median_year)
)
# Compare before and after for rows that were originally NaN
mask = df_before.isna()
pd.DataFrame({
    "Development Era": df.loc[mask, "Development Era"],
    "Before": df_before[mask],
    "After": df["Year of Construction"][mask]
})



,Development Era,Before,After
504,1930 to 1939 (Depression),NaN,1934.0
715,1885 to 1905 (Railway/Early Settlement),NaN,1895.0
749,1885 to 1905 (Railway/Early Settlement),NaN,1895.0
783,1875 to 1884 (Frontier),NaN,1879.0


In [4]:
df[df["Year of Construction"].isin([1000, 7000])]
yoc_mode = df["Year of Construction"].mode()[0]
df["Year of Construction"] = df["Year of Construction"].replace({1000: yoc_mode , 7000: yoc_mode})


df["Resource Type"].fillna(df["Resource Type"].mode()[0], inplace=True)


C:\Users\canmo\AppData\Local\Temp\ipykernel_12844\1482549516.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["Resource Type"].fillna(df["Resource Type"].mode()[0], inplace=True)


In [8]:
bool_cols = ["Provincial", "Municipal", "Is Demolished", "Is Decommissioned"]
df[bool_cols] = df[bool_cols].apply(lambda col: col.map({"Yes": True, "No": False, "Legal Agreement": False}))
pd.set_option('future.no_silent_downcasting', True)
df[bool_cols] = df[bool_cols].fillna(False).infer_objects(copy=False)


df["Architectural Style"] = df["Architectural Style"].fillna("Unknown")
df["Significance Summary"] = df["Significance Summary"].fillna("Unknown")
df["Description"] = df["Heritage Value"].fillna("Unknown")
df["Character Defining Elements"] = df["Character Defining Elements"].fillna("Unknown")
df["Heritage Value"] = df["Heritage Value"].fillna("Unknown")



df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 982 entries, 0 to 981
Data columns (total 16 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   Name                         982 non-null    object
 1   Community                    982 non-null    object
 2   Ward                         982 non-null    int64 
 3   Resource Type                982 non-null    object
 4   Year of Construction         982 non-null    object
 5   Development Era              982 non-null    object
 6   Architectural Style          982 non-null    object
 7   Original Use                 982 non-null    object
 8   Provincial                   982 non-null    bool  
 9   Municipal                    982 non-null    bool  
 10  Is Demolished                982 non-null    bool  
 11  Is Decommissioned            982 non-null    bool  
 12  Significance Summary         982 non-null    object
 13  Description                  982 no

In [6]:
df.to_csv("trimmed-data-nlp.csv", index=False)